In [1]:
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
from PIL import Image, ImageFile
from sklearn.model_selection import KFold, train_test_split
import Python_Modules.HelperFunctions as HelperFunctions

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F  # Import this line
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import time
import gc
import random

os.chdir('/media/my_drives')
warnings.simplefilter(action='ignore', category=FutureWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = True

seed = 1
random.seed(seed)                     # Python random
np.random.seed(seed)                  # NumPy random
torch.manual_seed(seed)               # PyTorch CPU random
torch.cuda.manual_seed(seed)          # PyTorch GPU random
torch.cuda.manual_seed_all(seed)      # If using multiple GPUs
torch.backends.cudnn.deterministic = True  # Force cuDNN to be deterministic
torch.backends.cudnn.benchmark = False     # Disable cuDNN auto-tuner (more reproducible)
os.environ['PYTHONHASHSEED'] = str(seed)

In [2]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'image_path']
        label = int(self.dataframe.loc[idx, 'label'])
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Function to create DataLoader
def make_dataloader(train_df, test_df, BATCH_SIZE, ALL_CLASSES):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard imagenet normalization
    ])
    
    train_dataset = CustomImageDataset(train_df, transform)
    test_dataset = CustomImageDataset(test_df, transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, test_loader

class BenchmarkModel(nn.Module):
    def __init__(self, N_CLASSES):
        super(BenchmarkModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 112 * 112, 64)  # Adjust based on image size
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, N_CLASSES)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(-1, 64 * 112 * 112)  # Flatten
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)

class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.counter = 0
        self.best_acc = None
        self.best_model_state = None
        self.early_stop = False

    def __call__(self, test_accuracy, model):
        if self.best_acc is None or test_accuracy < self.best_acc:
            self.best_acc = test_accuracy
            self.best_model_state = model.state_dict()
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# Training loop
def train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs=10):
    best_epoch = 0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    early_stopping = EarlyStopping(3)
    model.to(device)
    
    best_accuracy = 0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_accuracy = correct / total * 100
        
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_accuracy = correct / total
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy*100:.2f}%")
        
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            best_epoch = epoch

        early_stopping(test_accuracy, model)
        if early_stopping.early_stop:
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    return model, best_accuracy, best_epoch

In [4]:
CSV_PATH = 'DATA4/max/Image_Benchmark/R1_Analytics/Results.csv'
#df = pd.DataFrame(columns=['Model', 'Dataset', 'N_Training_Samples', 'BATCH_SIZE', 'LEARNING_RATE', 'EPOCHS', 'Time_Seconds', 'Accuracy'])
df = pd.read_csv(CSV_PATH)
df.tail(2)

,Model,Dataset,N_Training_Samples,BATCH_SIZE,LEARNING_RATE,EPOCHS,Time_Seconds,Accuracy,DROPOUT,BIGRAM_BOOL,BINARY_BOOL,MAX_FEATURES
9079,ResNet,GenImgNet,200.0,32.0,0.00001,16.0,219.7108,0.389972,NaN,NaN,NaN,NaN
9080,ResNet,GenImgNet,200.0,64.0,0.00001,10.0,158.5038,0.348189,NaN,NaN,NaN,NaN


In [5]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

In [6]:
MODEL_NAME = 'Full-NN-Training'

INPUT_SHAPE = (224, 224, 3)

BATCH_SIZES = [16, 32, 64]
LEARNING_RATES = [0.01, 0.001, 1e-5]
EPOCHS = 32
N_SAMPLES_SIZES = [1000, 200]

In [7]:
for DATASET_NAME in datasets:
    for BATCH_SIZE in BATCH_SIZES:
        for LEARNING_RATE in LEARNING_RATES:
            for N_Samples in N_SAMPLES_SIZES:
                if (DATASET_NAME == 'AIDA') & (N_Samples == 1000):
                    continue
                
                if df.loc[((df.Model == MODEL_NAME) & (df.Dataset == DATASET_NAME) & (df.N_Training_Samples == N_Samples) & (df.BATCH_SIZE == BATCH_SIZE) & (df.LEARNING_RATE == LEARNING_RATE))].shape[0] == 0:
                    
                    print(MODEL_NAME, ": ", DATASET_NAME, BATCH_SIZE, LEARNING_RATE)
    
                    DATASET_CSV = pd.read_csv(f"DATA4/max/Image_Benchmark/R1_Analytics/CSV_Training_{N_Samples}/{DATASET_NAME}.csv")
                    DATASET_CSV_HOLDOUT = pd.read_csv(f"DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET_NAME}.csv")
                    train_labels = set(DATASET_CSV['label'])
                    DATASET_CSV_HOLDOUT = DATASET_CSV_HOLDOUT[DATASET_CSV_HOLDOUT['label'].isin(train_labels)]
                    label_to_int = {label: idx for idx, label in enumerate(sorted(train_labels))}
                    DATASET_CSV_HOLDOUT = DATASET_CSV_HOLDOUT.reset_index(drop=True)
                    DATASET_CSV['label'] = DATASET_CSV['label'].map(label_to_int).astype(str)
                    DATASET_CSV_HOLDOUT['label'] = DATASET_CSV_HOLDOUT['label'].map(label_to_int).astype(str)
                    
                    class_names = list(DATASET_CSV['label'].unique())
                    n_classes = len(class_names)
                    train_loader, test_loader = make_dataloader(DATASET_CSV, DATASET_CSV_HOLDOUT, BATCH_SIZE, class_names)
                    
                    model = BenchmarkModel(N_CLASSES=n_classes)
                    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
                    criterion = nn.CrossEntropyLoss()
                    
                    start_time = time.time()
                    model, best_accuracy, best_epoch = train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs=EPOCHS)
                    total_time = np.round(time.time() - start_time, 2)
                    
                    # Save results
                    i = df.shape[0]
                    df.at[i, 'Model'] = MODEL_NAME
                    df.at[i, 'Dataset'] = DATASET_NAME
                    df.at[i, 'N_Training_Samples'] = N_Samples
                    df.at[i, 'BATCH_SIZE'] = BATCH_SIZE
                    df.at[i, 'LEARNING_RATE'] = LEARNING_RATE
                    df.at[i, 'EPOCHS'] = best_epoch+1
                    df.at[i, 'Time_Seconds'] = total_time
                    df.at[i, 'Accuracy'] = best_accuracy
                    df.to_csv(CSV_PATH, index=False)

                    gc.collect()

Full-NN-Training :  GenImgNet 16 0.01
Epoch 1/32, Train Loss: 1.2088, Train Accuracy: 34.00%, Test Accuracy: 32.87%
Epoch 2/32, Train Loss: 1.2197, Train Accuracy: 34.00%, Test Accuracy: 32.87%
Epoch 3/32, Train Loss: 1.2149, Train Accuracy: 34.00%, Test Accuracy: 32.87%
Epoch 4/32, Train Loss: 1.2053, Train Accuracy: 34.00%, Test Accuracy: 32.87%
Full-NN-Training :  GenImgNet 16 0.001
Epoch 1/32, Train Loss: 1.2032, Train Accuracy: 33.00%, Test Accuracy: 32.87%
Epoch 2/32, Train Loss: 1.2101, Train Accuracy: 34.00%, Test Accuracy: 32.87%
Epoch 3/32, Train Loss: 1.2101, Train Accuracy: 34.00%, Test Accuracy: 32.87%
Epoch 4/32, Train Loss: 1.2149, Train Accuracy: 34.00%, Test Accuracy: 32.87%
Full-NN-Training :  GenImgNet 16 1e-05
Epoch 1/32, Train Loss: 1.0466, Train Accuracy: 43.50%, Test Accuracy: 35.10%
Epoch 2/32, Train Loss: 0.8854, Train Accuracy: 72.50%, Test Accuracy: 39.00%
Epoch 3/32, Train Loss: 0.8087, Train Accuracy: 75.50%, Test Accuracy: 32.87%
Epoch 4/32, Train Loss: 0.